# Vector Quantization (VQ-VAE) with Straight-Through Gradients

**Difficulty**: 🟡 Medium

---

### Problem Statement

A **VQ-VAE** (van den Oord, Vinyals & Kavukcuoglu, NeurIPS 2017, *Neural Discrete Representation Learning*) replaces the Gaussian latent of a VAE (see `v4/generative/vae`) with a **discrete** one. The encoder produces a grid of vectors $z_e(x)$ of shape $(B, D, H, W)$, and the $D$-dimensional vector at every position is snapped to its nearest entry of a learned **codebook** $\{e_k\}_{k=1}^{K}$:

$$k_{bhw} = \arg\min_k \big\lVert z_e(x)_{b,:,h,w} - e_k \big\rVert_2, \qquad z_q(x)_{b,:,h,w} = e_{k_{bhw}}.$$

The grid of integers $k_{bhw}$ is the image's **token sequence**. VQGAN (Esser et al. 2021) trains a transformer over such tokens, MaskGIT and Muse predict them in parallel, Parti uses a ViT-VQGAN tokenizer, and neural audio codecs (SoundStream, EnCodec) stack several quantizers (residual VQ). All of them rely on the three pieces you will implement:

1. **Straight-through gradients.** The argmin has no useful gradient, so the forward pass uses $z_q$ while the backward pass copies the decoder's gradient to $z_e$ unchanged: $z_q^{\text{st}} = z_e + \mathrm{sg}[z_q - z_e]$, where $\mathrm{sg}$ (stop-gradient) is `.detach()`.
2. **Codebook and commitment losses** (the paper's eq. 3 without its reconstruction term), with the paper's $\beta = 0.25$:
$$\mathcal L_{\text{VQ}} = \underbrace{\big\lVert \mathrm{sg}[z_e] - e \big\rVert_2^2}_{\text{codebook term: moves } e} \;+\; \beta\,\underbrace{\big\lVert z_e - \mathrm{sg}[e] \big\rVert_2^2}_{\text{commitment term: moves } z_e}.$$
3. **An EMA codebook** (the paper's Appendix A.1): instead of the codebook term, move each code to an exponential moving average of the encoder vectors assigned to it, which is online k-means.

You will also compute the **codebook perplexity**, the standard diagnostic for dead codes and codebook collapse.

**Conventions (graded)**

| | |
|---|---|
| input | channel-first `z` of shape `(B, D, H, W)`; the $D$ values at one `(b, h, w)` form one vector |
| reduction | each loss term is a **mean over all $B\cdot D\cdot H\cdot W$ elements** (`F.mse_loss`). This is the paper's per-position squared norm divided by $D$ and averaged over positions, as in DeepMind Sonnet's `VectorQuantizer` (`tf.reduce_mean`) |
| EMA | plain EMAs as written in the paper's Appendix A.1 (no bias correction), plus the Laplace smoothing of Sonnet's `VectorQuantizerEMA` (equations below) |

---

### Requirements

1. **`VectorQuantizer(num_embeddings, embedding_dim, beta=0.25)`**, an `nn.Module` with
   - `self.embedding = nn.Embedding(num_embeddings, embedding_dim)`: the trainable $K \times D$ codebook (initialization is not graded; the reference uses $U(-1/K, 1/K)$);
   - `forward(z)` → `(z_q, loss, indices)` for `z` of shape `(B, D, H, W)`:
     - `indices`: shape `(B, H, W)`, dtype `torch.long`, the Euclidean-nearest code at every position, found **without a Python loop** over positions or codes;
     - `z_q`: shape `(B, D, H, W)` with value $e_{\text{indices}}$ at every position, returned as `z + (z_q - z).detach()`, so that $\partial z_q / \partial z = I$ and **no** gradient reaches the codebook through `z_q`;
     - `loss` (a scalar tensor) $= \mathrm{mean}\big((z_q - \mathrm{sg}[z])^2\big) + \beta\cdot\mathrm{mean}\big((z - \mathrm{sg}[z_q])^2\big)$;
     - do not modify `z` in place.
2. **`VectorQuantizerEMA(num_embeddings, embedding_dim, beta=0.25, decay=0.99, eps=1e-5)`**: same `forward` signature and outputs, except that
   - the codebook `self.embedding` (an `nn.Embedding`) is **frozen**: `self.embedding.weight.requires_grad_(False)`;
   - it has two buffers (`register_buffer`): `ema_cluster_size` of shape `(K,)` initialized to **ones**, and `ema_embed_sum` of shape `(K, D)` initialized to a **clone** of the initial codebook, so the state is consistent from the start ($m_k / \tilde N_k = e_k$);
   - `forward` quantizes with the codebook **as it is at the start of the call**, and `loss` $= \beta\cdot\mathrm{mean}\big((z - \mathrm{sg}[z_q])^2\big)$ (commitment only, since the codebook has no loss);
   - then, **only when `self.training`** and under `torch.no_grad()`, with $\gamma$ = `decay`, $\epsilon$ = `eps`, and the batch's $BHW$ vectors $z_j$ assigned to codes $k_j$:
   $$n_k = \#\{j : k_j = k\}, \qquad s_k = \sum_{j:\,k_j = k} z_j,$$
   $$N_k \leftarrow \gamma\, N_k + (1-\gamma)\, n_k, \qquad m_k \leftarrow \gamma\, m_k + (1-\gamma)\, s_k,$$
   $$\tilde N_k = \frac{N_k + \epsilon}{\sum_{k'} N_{k'} + K\epsilon}\,\sum_{k'} N_{k'}, \qquad e_k \leftarrow \frac{m_k}{\tilde N_k}.$$
   `ema_cluster_size` stores the **raw** $N$ (the smoothed $\tilde N$ is a local copy used only for the division, as in Sonnet, rosinality's and lucidrains' ports; some ports write $\tilde N$ back into the buffer, which is graded as wrong) and `ema_embed_sum` stores $m$;
   - in eval mode, nothing changes.
3. **`codebook_perplexity(indices, num_embeddings)`** → a scalar tensor $\exp\big(-\sum_k p_k \log p_k\big)$, where $p_k$ is the fraction of all entries of `indices` (any shape) equal to $k$, with $0 \log 0 = 0$. It is $K$ when all codes are used equally and $1$ when a single code is used.

---

### Constraints

- ✅ `nn.Embedding`, `F.one_hot`, `F.mse_loss`, `torch.cdist`, `torch.bincount` and `index_add_` are all fine.
- ❌ No Python loop over positions or codes, either in the nearest-code search or in the EMA statistics.
- ❌ No VQ libraries (`vector-quantize-pytorch`, Sonnet, …).
- ❌ No `.item()` inside the loss: it must stay differentiable.

---

<details>
  <summary>💡 Hint</summary>

  - `z.reshape(-1, D)` on a channel-first tensor does **not** give one vector per position. Which dimension has to be last before you flatten?
  - $\lVert z - e\rVert^2 = \lVert z\rVert^2 - 2\,z\cdot e + \lVert e\rVert^2$ turns the all-pairs distance into one matrix multiply. Dropping $\lVert e\rVert^2$ is a classic bug.
  - The codebook and commitment terms have the **same value**; they differ only in which side is detached. So a misplaced β is invisible in the loss value, and only a gradient test catches it.
  - Work out the value of `z + (z_q - z).detach()` and its derivative with respect to `z`. Then do the same for `z_q + (z - z_q).detach()`.
  - For the EMA, `F.one_hot(indices, K)` turns the assignments into a matrix: its column sums are $n$, and its transpose times the flattened vectors is $s$.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
# A 7-code, 3-D codebook whose codes have different norms and are >= 3 apart, so a vector
# planted at a code + small noise has an unambiguous nearest code.
CODEBOOK = torch.tensor([[0.0, 0.0, 0.0], [3.0, 0.0, 0.0], [0.0, 3.5, 0.0], [0.0, 0.0, -3.0],
                         [3.0, 3.0, 3.0], [-3.0, 1.0, 0.0], [-6.0, -6.0, 6.0]])


def planted_batch(b=2, h=4, w=5, noise=0.3, seed=0):
    # z of shape (b, 3, h, w) whose vector at each position is CODEBOOK[k] + noise.
    # Returns (z, true_indices of shape (b, h, w)).
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(0, len(CODEBOOK), (b, h, w), generator=g)
    z = CODEBOOK[idx].permute(0, 3, 1, 2) + noise * torch.randn(b, 3, h, w, generator=g)
    return z, idx


def brute_force_indices(z, codebook):
    # Reference nearest-code search: explicit (z - e)^2 sums in float64. z: (B, D, H, W) -> (B, H, W).
    diff = z.double().permute(0, 2, 3, 1)[..., None, :] - codebook.double()   # (B, H, W, K, D)
    return diff.pow(2).sum(-1).argmin(-1)


z, true_idx = planted_batch()
print(f"z {tuple(z.shape)} (B, D, H, W); codebook {tuple(CODEBOOK.shape)} (K, D)")

In [ ]:
class VectorQuantizer(nn.Module):
    """Nearest-neighbour vector quantizer with codebook + commitment losses and
    straight-through gradients (van den Oord et al. 2017)."""

    def __init__(self, num_embeddings, embedding_dim, beta=0.25):
        """
        Args:
            num_embeddings: K, the number of codes.
            embedding_dim:  D, the size of each code (= the channel count of z).
            beta:           weight of the commitment term.

        Keep the codebook in self.embedding = nn.Embedding(num_embeddings, embedding_dim).
        """
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z):
        """
        Args:
            z: encoder output, shape (B, D, H, W), channel-first.

        Returns:
            z_q:     (B, D, H, W); value codebook[indices] laid out like z, gradient
                     straight-through to z (z + (z_q - z).detach()).
            loss:    scalar tensor, mse(z_q, sg[z]) + beta * mse(z, sg[z_q]) (means over all elements).
            indices: (B, H, W) torch.long, the nearest code at every position.
        """
        # TODO
        raise NotImplementedError

In [ ]:
class VectorQuantizerEMA(nn.Module):
    """Vector quantizer whose codebook is an exponential moving average of the encoder
    vectors assigned to each code (Appendix A.1 of the paper, Laplace smoothing as in Sonnet)."""

    def __init__(self, num_embeddings, embedding_dim, beta=0.25, decay=0.99, eps=1e-5):
        """
        Args:
            num_embeddings: K.   embedding_dim: D.
            beta:  commitment weight.   decay: EMA decay gamma.   eps: Laplace smoothing epsilon.

        self.embedding = nn.Embedding(K, D) with requires_grad False, plus buffers
        ema_cluster_size (K,) = ones and ema_embed_sum (K, D) = clone of the initial codebook.
        """
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z):
        """
        Args:
            z: (B, D, H, W).

        Returns:
            (z_q, loss, indices) as in VectorQuantizer, computed with the codebook as it is at
            the start of the call; loss = beta * mse(z, sg[z_q]). In training mode, afterwards
            update ema_cluster_size, ema_embed_sum and the codebook (no autograd history).
            ema_cluster_size keeps the raw EMA N; the Laplace-smoothed N~ is only a local
            copy used to divide.
        """
        # TODO
        raise NotImplementedError

In [ ]:
def codebook_perplexity(indices, num_embeddings):
    """
    Args:
        indices:        integer code indices, any shape (e.g. (B, H, W)).
        num_embeddings: K.

    Returns:
        Scalar tensor exp(-sum_k p_k log p_k), p_k = fraction of entries equal to k, 0 log 0 = 0.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Test 1: shapes, nearest codes on planted inputs, and agreement with brute force
torch.manual_seed(0)
vq = VectorQuantizer(num_embeddings=7, embedding_dim=3)
with torch.no_grad():
    vq.embedding.weight.copy_(CODEBOOK)
z, true_idx = planted_batch()
z_q, loss, idx = vq(z)
assert z_q.shape == z.shape and idx.shape == (2, 4, 5), (z_q.shape, idx.shape)
assert idx.dtype == torch.long and loss.dim() == 0, (idx.dtype, loss.shape)
assert torch.equal(idx, true_idx), "planted vectors must map to their own code (permute before you flatten)"
assert torch.allclose(z_q, CODEBOOK[idx].permute(0, 3, 1, 2), atol=1e-6), \
    "z_q must hold codebook[indices], laid out channel-first"

vq16 = VectorQuantizer(16, 4)
with torch.no_grad():
    vq16.embedding.weight.copy_(torch.randn(16, 4))
z16 = 1.5 * torch.randn(3, 4, 6, 6)
assert torch.equal(vq16(z16)[2], brute_force_indices(z16, vq16.embedding.weight)), \
    "indices must match the brute-force argmin"
print("Test 1 passed: shapes, planted indices, brute-force agreement")

In [ ]:
# Test 2: straight-through gradient, loss value, and which loss term feeds which gradient
vq = VectorQuantizer(7, 3, beta=0.25)
with torch.no_grad():
    vq.embedding.weight.copy_(CODEBOOK)
z, _ = planted_batch(noise=0.6, seed=1)
z.requires_grad_(True)

# (a) through z_q alone: d z_q / d z = I and nothing reaches the codebook
z_q, loss, idx = vq(z)
g = torch.randn_like(z)
(z_q * g).sum().backward()
assert torch.allclose(z.grad, g), "d z_q / d z must be the identity"
assert vq.embedding.weight.grad is None or (vq.embedding.weight.grad == 0).all(), \
    "z_q must not send gradient to the codebook"

# (b) the loss alone: value (1 + beta) * mse; the encoder gets only beta * commitment,
#     the codebook only the codebook term
z.grad, vq.embedding.weight.grad = None, None
z_q, loss, idx = vq(z)
zq = CODEBOOK[idx].permute(0, 3, 1, 2)
mse = ((zq - z.detach()) ** 2).mean()
assert torch.allclose(loss, 1.25 * mse), f"loss {loss.item():.5f} != (1 + beta) * mse = {1.25 * mse.item():.5f}"
loss.backward()
n = z.numel()
assert torch.allclose(z.grad, 0.25 * 2 * (z.detach() - zq) / n, atol=1e-7), \
    "the encoder must get only beta * d(commitment)/dz"
want = torch.zeros(7, 3).index_add_(0, idx.flatten(),
                                    (zq - z.detach()).permute(0, 2, 3, 1).reshape(-1, 3)) * 2 / n
assert torch.allclose(vq.embedding.weight.grad, want, atol=1e-7), "the codebook must get only d(codebook term)/de"
print("Test 2 passed: straight-through gradient, loss value, isolated gradients")

In [ ]:
# Test 3: one EMA step against the equations, then eval mode
decay, eps = 0.8, 0.1                                    # a large eps makes the smoothing visible
ema = VectorQuantizerEMA(7, 3, beta=0.25, decay=decay, eps=eps)
assert not ema.embedding.weight.requires_grad, "the EMA codebook must be frozen"
assert {"ema_cluster_size", "ema_embed_sum"} <= set(dict(ema.named_buffers())), "register both EMA buffers"
assert torch.allclose(ema.ema_embed_sum / ema.ema_cluster_size[:, None], ema.embedding.weight), \
    "fresh state: m / N must equal the codebook"

N0 = torch.tensor([2.0, 0.5, 1.0, 3.0, 0.2, 1.5, 0.0])
M0 = torch.randn(7, 3)
with torch.no_grad():
    ema.embedding.weight.copy_(CODEBOOK)
    ema.ema_cluster_size.copy_(N0)
    ema.ema_embed_sum.copy_(M0)
z, _ = planted_batch(seed=2)
z_q, loss, idx = ema(z)
assert torch.allclose(z_q, CODEBOOK[idx].permute(0, 3, 1, 2), atol=1e-6), \
    "quantize with the codebook from BEFORE the update"
assert torch.allclose(loss, 0.25 * ((z_q - z) ** 2).mean()), "EMA loss = beta * commitment"

flat = z.permute(0, 2, 3, 1).reshape(-1, 3)
n = torch.bincount(idx.flatten(), minlength=7).float()
s = torch.zeros(7, 3).index_add_(0, idx.flatten(), flat)
N1 = decay * N0 + (1 - decay) * n
M1 = decay * M0 + (1 - decay) * s
N_smooth = (N1 + eps) / (N1.sum() + 7 * eps) * N1.sum()
assert torch.allclose(ema.ema_cluster_size, N1), "ema_cluster_size must hold the raw EMA of the counts"
assert torch.allclose(ema.ema_embed_sum, M1, atol=1e-6), "ema_embed_sum must hold the EMA of the assigned sums"
assert torch.allclose(ema.embedding.weight, M1 / N_smooth[:, None], atol=1e-5), \
    "codebook = m / (Laplace-smoothed N)"

ema.eval()
before = ema.embedding.weight.clone()
ema(planted_batch(seed=3)[0])
assert torch.equal(ema.embedding.weight, before), "eval mode must not update the codebook"
print("Test 3 passed: EMA update equations; eval mode is read-only")

In [ ]:
# Test 4: perplexity
assert abs(codebook_perplexity(torch.arange(8).repeat(4), 8).item() - 8) < 1e-5, "uniform over 8 codes -> 8"
one = codebook_perplexity(torch.zeros(2, 3, 3, dtype=torch.long), 8).item()
assert abs(one - 1) < 1e-5, f"a single code in use -> 1 (and no NaN for unused codes), got {one}"
p = torch.tensor([0.75, 0.25])
want = torch.exp(-(p * p.log()).sum()).item()
assert abs(codebook_perplexity(torch.tensor([2, 2, 2, 5]), 8).item() - want) < 1e-5
print(f"Test 4 passed: perplexity (a 0.75 / 0.25 split -> {want:.4f} effective codes)")

In [ ]:
# Dead codes and restarts (~1-2 s). Each 2-D point is a 1x1 "image" with D = 2 and the encoder
# is the identity, so only the codebook learns. All 32 codes start in a tiny box around the
# origin; the data lives on a ring of radius 3.
def ring(n, g):
    ang = torch.randint(0, 8, (n,), generator=g) * (2 * math.pi / 8)
    x = 3 * torch.stack([ang.cos(), ang.sin()], 1) + 0.3 * torch.randn(n, 2, generator=g)
    return x[:, :, None, None]                                  # (n, 2, 1, 1)


def fit(kind, restart=False, K=32, steps=300):
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(1)
    if kind == "grad":
        vq = VectorQuantizer(K, 2)
        opt = torch.optim.Adam(vq.parameters(), lr=0.05)
    else:
        vq = VectorQuantizerEMA(K, 2, decay=0.9)
    with torch.no_grad():                                       # same start whatever your init:
        vq.embedding.weight.uniform_(-1 / K, 1 / K)              # every code in a tiny box at the origin
        if kind == "ema":
            vq.ema_cluster_size.fill_(1.0)
            vq.ema_embed_sum.copy_(vq.embedding.weight)
    for _ in range(steps):
        z = ring(256, g)
        _, loss, _ = vq(z)
        if kind == "grad":
            opt.zero_grad()
            loss.backward()
            opt.step()
        if restart:                                             # re-seed codes nobody has used lately
            with torch.no_grad():
                dead = vq.ema_cluster_size < 0.05
                if dead.any():
                    pick = z.reshape(-1, 2)[torch.randint(0, len(z), (int(dead.sum()),), generator=g)]
                    vq.embedding.weight[dead] = pick
                    vq.ema_embed_sum[dead] = pick
                    vq.ema_cluster_size[dead] = 1.0
    vq.eval()
    with torch.no_grad():
        z = ring(4096, torch.Generator().manual_seed(2))
        z_q, _, idx = vq(z)
    return codebook_perplexity(idx, K).item(), idx.unique().numel(), ((z_q - z) ** 2).mean().item()


for name, kw in [("codebook loss (Adam)", dict(kind="grad")), ("EMA", dict(kind="ema")),
                 ("EMA + dead-code restarts", dict(kind="ema", restart=True))]:
    perp, used, err = fit(**kw)
    print(f"{name:26s} perplexity {perp:5.2f}   codes used {used:2d}/32   quantization mse {err:.4f}")
# Only the codes that happen to be nearest to some data point ever move; the rest stay dead in
# both the gradient and the EMA version (an unused code gets neither gradient nor statistics).
# Restarting dead codes at random encoder outputs brings every code into use and cuts the error.

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vq-vae", VectorQuantizer=VectorQuantizer, VectorQuantizerEMA=VectorQuantizerEMA,
      codebook_perplexity=codebook_perplexity)